# 1. What is Categorical Encoding?

### Concept & Definition
Categorical encoding is the process of converting qualitative text labels into numerical representations so machine learning algorithms can interpret them.

### Types of Categorical Variables:
- **Nominal:** No intrinsic order (e.g., `Gender`, `PaymentMethod`).
- **Ordinal:** Has a clear hierarchical order (e.g., `ContractType`: Month-to-month < One Year < Two Year).
- **Binary:** Exactly two mutually exclusive categories (e.g., `Yes`/`No`).

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("Cleaned_Validated_Data.csv")

# Identify categorical columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical Columns Found:", cat_cols)

Categorical Columns Found: ['CustomerID', 'Gender', 'ContractType', 'PaymentMethod', 'Churn']


C:\Users\abarn\AppData\Local\Temp\ipykernel_5376\2891650128.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()


# 2. Binary & Label Encoding

### Concept & Definition
- **Binary Encoding (0/1):** Maps binary categories directly to `0` and `1`.
- **Label Encoding:** Assigns a unique integer ($0, 1, 2, \dots$) to each category level.

### When to Use:
- Target variables (`Churn`: Yes=1, No=0) or tree algorithms.

### Risks:
- Linear models misinterpret integer label values as magnitude rankings ($2 > 1$).

In [2]:
from sklearn.preprocessing import LabelEncoder

df_label = df.copy()

# Target Encoding using LabelEncoder
le = LabelEncoder()
df_label["Churn_Encoded"] = le.fit_transform(df_label["Churn"].astype(str))

display(df_label[["Churn", "Churn_Encoded"]].head(5))

,Churn,Churn_Encoded
0,No,0
1,Yes,1
2,No,0
3,Yes,1
4,Yes,1


# 3. Ordinal Encoding

### Concept & Definition
Encodes categorical values into ordered numbers while explicitly preserving their real-world domain ranking.

from sklearn.preprocessing import OrdinalEncoder

df_ordinal = df.copy()

# Mapping ContractType explicitly
contract_order = [["Month-to-Month", "One Year", "Two Year"]]
oe = OrdinalEncoder(categories=contract_order)

df_ordinal["ContractType_Encoded"] = oe.fit_transform(df_ordinal[["ContractType"]].fillna("Month-to-Month"))

display(df_ordinal[["ContractType", "ContractType_Encoded"]].head(5))

# 4. One-Hot Encoding & Dummy Variables

### Concept & Definition
Creates separate binary column indicators ($0$ or $1$) for each unique category level. Setting `drop_first=True` avoids the **Dummy Variable Trap** (multicollinearity).

### When to Use:
- Low-cardinality nominal variables in linear models, SVMs, or neural networks.

In [5]:
# One-Hot Encoding using pandas get_dummies
df_ohe = pd.get_dummies(df, columns=["PaymentMethod"], drop_first=True, dtype=int)

print("Columns added post-OHE:")
display([c for c in df_ohe.columns if "PaymentMethod" in c])

Columns added post-OHE:


['PaymentMethod_Credit card',
 'PaymentMethod_Electronic check',
 'PaymentMethod_Mailed check']

# 5. Frequency Encoding

### Concept & Definition
Replaces each category level with its normalized frequency or absolute count within the dataset.

### Pros & Cons:
- **Pro:** Preserves dimensional size (adds zero new columns).
- **Con:** Two distinct categories with identical counts end up with identical encoded values.

In [6]:
# Frequency Encoding for PaymentMethod
df_freq = df.copy()

freq_map = (df_freq["PaymentMethod"].value_counts() / len(df_freq)).to_dict()
df_freq["PaymentMethod_Freq"] = df_freq["PaymentMethod"].map(freq_map)

display(df_freq[["PaymentMethod", "PaymentMethod_Freq"]].head(5))

,PaymentMethod,PaymentMethod_Freq
0,Credit card,0.237624
1,Mailed check,0.256436
2,Credit card,0.237624
3,Bank transfer,0.250495
4,Mailed check,0.256436


# 6. Target Encoding

### Concept & Definition
Replaces each category level with the mean of the target variable for that specific category.

### Risks:
- High risk of **Data Leakage** and overfitting if calculated without out-of-fold cross-validation.

In [7]:
# Target Encoding example (Mean Churn Rate per Payment Method)
df_target = df.copy()
df_target["Churn_Num"] = np.where(df_target["Churn"] == "Yes", 1, 0)

target_map = df_target.groupby("PaymentMethod")["Churn_Num"].mean().to_dict()
df_target["PaymentMethod_TargetEnc"] = df_target["PaymentMethod"].map(target_map)

display(df_target[["PaymentMethod", "PaymentMethod_TargetEnc"]].head(5))

,PaymentMethod,PaymentMethod_TargetEnc
0,Credit card,0.31250
1,Mailed check,0.30888
2,Credit card,0.31250
3,Bank transfer,0.27668
4,Mailed check,0.30888


# 7. Handling Unknown Categories & High Cardinality

### Strategies:
1. **Unknown Categories at Inference:** Use `OneHotEncoder(handle_unknown='ignore')` to assign zeroes to unseen test levels.
2. **High Cardinality (> 50 categories):** Group rare categories into an `"Other"` bin or use Frequency/Target Encoding to avoid high dimensional expansion.

In [8]:
from sklearn.preprocessing import OneHotEncoder

# Handling unknown test categories cleanly
ohe_robust = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe_robust.fit(df[["Gender"]].fillna("Unknown"))

# Transform sample with an unseen category
sample_test = pd.DataFrame({"Gender": ["Male", "Female", "Non-Binary-Unseen"]})
encoded_array = ohe_robust.transform(sample_test)

print("Robust OHE Output for Unseen Category sample:")
print(encoded_array)
print("Notebook 07 execution completed successfully!")

Robust OHE Output for Unseen Category sample:
[[0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
Notebook 07 execution completed successfully!
